# 01 · Data understanding and preprocessing

Goal: understand the VisDrone-VID frames before training and fix the choices used by every detector:
the validation split, frame subsampling, input resolution, anchor sizes and handling of ignored regions.
Runs on CPU.

In [ ]:
import os, urllib.request
# helper modules shared by all notebooks (data split, metric, submission format, torchvision training)
REPO = "https://raw.githubusercontent.com/MakcDutch/AI-homework2/main/"
for f in ["common.py", "tv_train.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(REPO + f, f)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
import common as C

ROOT = C.find_data_root()
FIG_DIR = "figures"; os.makedirs(FIG_DIR, exist_ok=True)
pd.set_option("display.width", 160)
print("data root:", ROOT)

## 1. Loading
Train and val come with COCO annotations; `iscrowd = 1` rows are ignored regions, not objects.

In [ ]:
frames = {s: C.list_frames(ROOT, s) for s in ["train", "val", "test"]}
coco = {s: C.load_coco_frames(ROOT, s) for s in ["train", "val"]}
test_meta = pd.read_csv(ROOT / "test_images.csv")

objs = pd.concat([coco[s][1].assign(split=s) for s in coco], ignore_index=True)
gt = objs[objs["iscrowd"] == 0].copy()
ign = objs[objs["iscrowd"] == 1].copy()

def sizes(df):
    return ", ".join(f"{w}x{h}" for w, h in sorted(set(zip(df["width"], df["height"]))))

overview = pd.DataFrame({
    s: dict(clips=frames[s]["clip"].nunique(), frames=len(frames[s]),
            objects=int((gt["split"] == s).sum()) if s != "test" else np.nan,
            ignored_regions=int((ign["split"] == s).sum()) if s != "test" else np.nan,
            image_sizes=sizes(coco[s][0] if s != "test" else test_meta))
    for s in ["train", "val", "test"]}).T
overview

## 2. Class balance

In [ ]:
cls = gt.groupby(["class", "split"]).size().unstack().reindex(C.CLASSES).fillna(0).astype(int)
share = (cls / cls.sum() * 100).round(2).add_suffix(" %")
display(pd.concat([cls, share], axis=1))

ax = cls.plot.bar(logy=True, figsize=(9, 3.5), title="Objects per class (log scale)")
ax.set_ylabel("objects"); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/class_balance.png", dpi=120); plt.show()

## 3. Object sizes
`size` = sqrt(w·h) in original pixels. The second table shows what remains of an object after the frame's
long side is resized to a given network input size: this is what fixes the input resolution.

In [ ]:
bins = [0, 8, 16, 32, 64, 96, np.inf]
labels = ["<8", "8-16", "16-32", "32-64", "64-96", ">96"]
gt["size_bin"] = pd.cut(gt["size"], bins, labels=labels, right=False)
print("share of objects by size, %")
display((pd.crosstab(gt["split"], gt["size_bin"], normalize="index") * 100).round(1))

gt["coco_area"] = pd.cut(gt["w"] * gt["h"], [0, 32**2, 96**2, np.inf], labels=["small", "medium", "large"], right=False)
print("COCO area groups, %")
display((pd.crosstab(gt["split"], gt["coco_area"], normalize="index") * 100).round(1))

print("median box per class, px")
display(gt.groupby("class")[["w", "h", "size"]].median().reindex(C.CLASSES).round(1))

long_side = np.maximum(gt["width"], gt["height"])
rows = []
for imgsz in [640, 1024, 1280, 1536]:
    s = gt["size"] * imgsz / long_side
    rows.append(dict(input_size=imgsz, median_px=s.median(), below_8px_pct=(s < 8).mean() * 100,
                     below_16px_pct=(s < 16).mean() * 100))
print("object size after resizing the long side")
display(pd.DataFrame(rows).round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for s in ["train", "val"]:
    axes[0].hist(gt.loc[gt["split"] == s, "size"], bins=np.logspace(0, 3, 60), histtype="step", density=True, label=s)
axes[0].set_xscale("log"); axes[0].set_xlabel("sqrt(w*h), px"); axes[0].set_title("Object size"); axes[0].legend()
axes[1].hist(np.log2(gt["w"] / gt["h"]), bins=60)
axes[1].set_xlabel("log2(w / h)"); axes[1].set_title("Aspect ratio")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/object_sizes.png", dpi=120); plt.show()

## 4. Scene density
Objects per frame overall and per clip. Clips differ a lot, which is why the split is by clip.

In [ ]:
keys = pd.concat([coco[s][0][["key"]].assign(split=s) for s in coco])
per_frame = keys.merge(gt.groupby("key").size().rename("n").reset_index(), on="key", how="left").fillna({"n": 0})
display(per_frame.groupby("split")["n"].describe(percentiles=[0.5, 0.9, 0.99]).round(1))

per_frame["clip"] = per_frame["key"].str.split("/").str[0]
clip_stats = per_frame.groupby(["split", "clip"])["n"].agg(frames="size", mean_objects="mean", max_objects="max")
top_class = gt.groupby(["split", "clip"])["class"].agg(lambda c: c.value_counts().index[0])
med_size = gt.groupby(["split", "clip"])["size"].median()
clip_stats = clip_stats.join(top_class.rename("top_class")).join(med_size.rename("median_size_px")).round(1)
display(clip_stats.sort_values("mean_objects", ascending=False).head(15))
display(clip_stats.loc["val"])

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.5))
order = clip_stats.loc["train"].sort_values("mean_objects").index
ax.boxplot([per_frame.loc[per_frame["clip"] == c, "n"] for c in order], showfliers=False)
ax.set_xticks([]); ax.set_xlabel("train clips (sorted by mean density)"); ax.set_ylabel("objects per frame")
ax.set_title("Objects per frame by clip")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/density_by_clip.png", dpi=120); plt.show()

## 5. Ignored regions, occlusion and truncation

In [ ]:
def union_share(d, cell=4):
    # share of the frame covered by the union of ignored regions (rasterised on a 4 px grid; regions overlap)
    W, H = int(d["width"].iloc[0]), int(d["height"].iloc[0])
    m = np.zeros((H // cell + 1, W // cell + 1), bool)
    for x1, y1, x2, y2 in (np.c_[d["x"], d["y"], d["x"] + d["w"], d["y"] + d["h"]] / cell).round().astype(int):
        m[max(y1, 0):y2 + 1, max(x1, 0):x2 + 1] = True
    return m.mean()

ign_frames = pd.Series({k: union_share(d) for k, d in ign.groupby(["split", "key"])})
summary = pd.DataFrame({
    "frames_with_ignore_pct": ign.groupby("split")["key"].nunique() / keys.groupby("split").size() * 100,
    "median_area_covered_pct": ign_frames.groupby(level=0).median() * 100,
    "max_area_covered_pct": ign_frames.groupby(level=0).max() * 100})
display(summary.round(1))

# objects whose box lies mostly inside an ignored region (they would be painted over by masking)
ign_by_key = {k: g for k, g in ign[ign["split"] == "train"].groupby("key")}
inside = []
for key, g in gt[gt["split"] == "train"].groupby("key"):
    r = ign_by_key.get(key)
    if r is not None:
        b = g[["x", "y", "w", "h"]].to_numpy(); rr = r[["x", "y", "w", "h"]].to_numpy()
        inside.append(C.inside_fraction(np.c_[b[:, :2], b[:, :2] + b[:, 2:]], np.c_[rr[:, :2], rr[:, :2] + rr[:, 2:]]))
inside = np.concatenate(inside) if inside else np.zeros(0)
print(f"train objects >50% inside an ignored region: {(inside > 0.5).sum()} of {(gt['split'] == 'train').sum()}")

for col in ["occlusion", "truncation"]:
    if col in gt:
        print(col, "share, %"); display((pd.crosstab(gt["split"], gt[col], normalize="index") * 100).round(1))

## 6. Temporal redundancy → frame subsampling
For the same `track_id`, how much does a box move after `lag` frames? High IoU means the frames are near-duplicates
and training on all of them mostly repeats the same information.

In [ ]:
tr = gt.loc[gt["split"] == "train", ["clip", "frame", "track_id", "x", "y", "w", "h"]]
rows = []
for lag in [1, 2, 3, 5, 10, 20, 50]:
    m = tr.merge(tr.assign(frame=tr["frame"] - lag), on=["clip", "track_id", "frame"], suffixes=("", "_n"))
    ix = (np.minimum(m["x"] + m["w"], m["x_n"] + m["w_n"]) - np.maximum(m["x"], m["x_n"])).clip(0)
    iy = (np.minimum(m["y"] + m["h"], m["y_n"] + m["h_n"]) - np.maximum(m["y"], m["y_n"])).clip(0)
    iou = ix * iy / (m["w"] * m["h"] + m["w_n"] * m["h_n"] - ix * iy)
    rows.append(dict(lag=lag, tracks_still_present_pct=len(m) / len(tr) * 100, median_iou=iou.median(),
                     iou_above_0_7_pct=(iou > 0.7).mean() * 100))
display(pd.DataFrame(rows).round(2))

## 7. Anchor shapes from the data (k-means, YOLOv2 idea)
Boxes are rescaled to a 1024 px long side. Clustering is done in log(w, h) space so that small boxes are not
dominated by large ones. These sizes are reused for the anchor-based detectors.

In [ ]:
from sklearn.cluster import KMeans

train_gt = gt[gt["split"] == "train"]
scale = 1024 / np.maximum(train_gt["width"], train_gt["height"])
wh = np.c_[train_gt["w"] * scale, train_gt["h"] * scale]
wh = wh[np.random.default_rng(C.SEED).permutation(len(wh))[:200_000]]  # a 200k-box sample is enough for 9 centres
km = KMeans(n_clusters=9, n_init=10, random_state=C.SEED).fit(np.log(wh))
anchors = np.exp(km.cluster_centers_)
anchors = anchors[np.argsort(anchors.prod(1))]
display(pd.DataFrame({"w": anchors[:, 0], "h": anchors[:, 1], "size": np.sqrt(anchors.prod(1)),
                      "aspect_w/h": anchors[:, 0] / anchors[:, 1]}).round(1))
print("size quantiles at 1024 px:", np.percentile(np.sqrt(wh.prod(1)), [5, 25, 50, 75, 95]).round(1))

## 8. Checking the labels on frames
Ground truth from the YOLO `.txt` files (class colours), ignored regions (red hatch). The right column shows the
same frame after masking ignored regions, which is what the detectors are trained on.

In [ ]:
busy = per_frame[per_frame["split"] == "train"].sort_values("n", ascending=False).drop_duplicates("clip")
sample_keys = busy["key"].iloc[[0, len(busy) // 3, 2 * len(busy) // 3]].tolist()

fig, axes = plt.subplots(len(sample_keys), 2, figsize=(14, 4.2 * len(sample_keys)))
for row, key in zip(axes, sample_keys):
    C.draw_frame(row[0], ROOT, "train", key, title=f"{key}: labels + ignored regions")
    clip, stem = key.split("/")
    img = cv2.imread(str(ROOT / "train" / "images" / clip / f"{stem}.jpg"))
    H, W = img.shape[:2]
    regions = C.xywhn_to_xyxy(C.read_txt(str(ROOT / "train" / "ignore" / clip / f"{stem}.txt")).reshape(-1, 4), W, H)
    row[1].imshow(cv2.cvtColor(C.mask_regions(img, regions), cv2.COLOR_BGR2RGB)); row[1].axis("off")
    row[1].set_title("after masking ignored regions", fontsize=9)
for c, name in enumerate(C.CLASSES):
    axes[0, 0].plot([], [], color=np.array(C.COLORS[c]) / 255, label=name)
axes[0, 0].legend(loc="upper right", fontsize=7)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/label_check.png", dpi=110); plt.show()

## 9. Test set
The test clips should look like train/val (same resolutions, similar clip length).

In [ ]:
print("frames per clip")
display(pd.DataFrame({
    "train": frames["train"].groupby("clip").size().describe(),
    "val": frames["val"].groupby("clip").size().describe(),
    "test": test_meta.groupby("clip").size().describe()}).round(1))
print("test image sizes:", sizes(test_meta))

## Observations used later

TODO: write the conclusions yourself (class imbalance, share of small objects, chosen input size,
frame step, anchor sizes, ignored-region handling).